James Caldwell <br>
UVA IRA, 11/11/26 <br>

This Python script automates the extraction of error tables from the SCHEV Institutional Portal and merges them with a VCSIN-to-SSID mapping file. The final result is exported to an Excel workbook with each error code on a separate sheet.

Features: <br>
Opens the SCHEV portal in a Chrome browser via Selenium.

Allows manual login for secure access.

Extracts links corresponding to error codes.

Follows each link to retrieve error tables.

Merges error tables with a local VCSIN-to-SSID Excel mapping.

Saves all merged tables to a single Excel workbook with meaningful sheet names.

In [5]:
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from bs4 import BeautifulSoup
import time
from urllib.parse import urljoin
from io import StringIO
import re
from dotenv import load_dotenv
import os

# os.chdir(r'filepath of this script here')

load_dotenv()
# Variables to change
reporting_year = os.getenv('reporting_year') 
url = "https://portals.schev.edu/institutions/PUBLIC/UVA/Viewerrorsummary.asp?tablename=fa&repyear="+reporting_year
# reporting_year = '2627'
# url = "https://portals.schev.edu/institutions/PUBLIC/UVA/Viewerrorsummary.asp?tablename=hc&repyear="+reporting_year
load_dotenv()
VCSIN_to_SSID_path = os.getenv('VCSIN_to_SSID_path') 
VCSIN_to_SSID_path_inactive = os.getenv('VCSIN_to_SSID_path_inactive') 
chrome_driver_path = os.getenv('chrome_driver_path') # This won't run on the V: drive. Put in folder on personal computer. Download from: https://googlechromelabs.github.io/chrome-for-testing/
excel_output_path = os.getenv('excel_output_path') 
if os.path.exists(excel_output_path):
        os.remove(excel_output_path) # Remove existing file if it exists to avoid overwrite issues

options = Options()
driver = webdriver.Chrome(options=options) 

# Open the login page
driver.get(url)

print("Browser opened. Please log in manually.")
version_number = input("Type input number and ENTER here *after* you have logged in successfully...")

# After you log in, grab the page
driver.get(url)
html = driver.page_source
mysoup = BeautifulSoup(html, 'html.parser')

# Load VCSIN to SSID mapping file
print("Loading VCSIN to SSID mapping file...")
VCSIN_to_SSID = pd.read_excel(VCSIN_to_SSID_path)
VCSIN_to_SSID_inactive = pd.read_excel(VCSIN_to_SSID_path_inactive)
# Stack them
VCSIN_to_SSID = pd.concat([VCSIN_to_SSID, VCSIN_to_SSID_inactive], ignore_index=True, sort=False)

# Load meta data file (This has error descriptions and links)
print("Loading meta data file...")
meta_data_path = os.getenv('meta_data_path')
meta_data = pd.read_excel(meta_data_path)

# === Extract all links ===
links = []
for a in mysoup.find_all('a', href=True):
    # print(mysoup)
    # print('end of first soup')
    href = a['href']
    # Skip empty, anchor, or javascript links
    if href.startswith('#') or href.lower().startswith('javascript'):
        continue
    # Convert relative URLs to absolute
    full_link = href if href.startswith('http') else driver.current_url.rsplit('/', 1)[0] + '/' + href
    if 'viewError' in full_link:  # filter for relevant links
        links.append(full_link)

print(f"\nFound {len(links)} error code links:")


soup_count = 0

all_error_tables_for_excel = []
all_error_table_names_for_excel = []

meta_check_df = pd.DataFrame({'url': links})

meta_check_df['Err'] = meta_check_df['url'].str.extract(r'Err=([^&]+)')
meta_check_df['SCHEV Definition'] = None
meta_check_df['SCHEV Warning'] = None
no_errors = True #meta_check_df['Err'].isin(meta_data['ErrCode']).all() # True if all error codes in links are in meta_data, otherwise False
if no_errors: # Check that all error codes in links are in meta_data. 
    print("All error codes found in meta_data.")
    # # === Visit each link ===
    for i, link in enumerate(links, start=1): # replace "links" with links[:3] for shorter testing
        print(f"\nVisiting link {i}/{len(links)}: {link}")
        try:
            driver.get(link)
            time.sleep(1)  # wait for page to load; adjust as needed
            
            page_html = driver.page_source
            page_soup = BeautifulSoup(page_html, 'html.parser')
            if soup_count == 0:
                # print(page_soup)
                soup_count += 1
            # Find the <script> tag containing the specific substring for error table url
            link_tag = page_soup.find(src=lambda value: value and "ErrorsList.asp" in value)
            if link_tag:
                
                ## save Error Name for excel sheet name
                match = re.search(r'Errcode=([^&"]+)', page_html)

                if match:
                    error_code = match.group(1)
                    # print(f'Error Code: {error_code}')
                    all_error_table_names_for_excel.append(error_code)  # error_code = match.group(1)

                print(error_code)
                # if error_code == 'AFE010E02':
                #     print(page_soup)
 
                ## Follow error table URL and save
                relative_url = link_tag['src']
                # print("Relative URL:", relative_url)
                
                ## Save the error "definition" for reference
                definition_label = page_soup.find(
                    'b',
                    string=lambda text: text and 'Definition:' in text
                )
                if definition_label:
                    SCHEV_definition = definition_label.find_parent('td').find_next_sibling('td').get_text(
                        " ", strip=True
                    )
                    meta_check_df.loc[i-1, 'SCHEV Definition'] = SCHEV_definition
                    # print(definition)

                ## Save error message for reference
                warning_text = page_soup.find(
                    string=lambda text: text and 'warning(s)' in text
                )
                if warning_text:
                    warning = warning_text.parent.parent.get_text(" ", strip=True)
                    meta_check_df.loc[i-1, 'SCHEV Warning'] = warning
                else: # Errors have slightly different syntax than the warnings
                    # print(error_code + ' made it to error(s) check!')
                    error_text = page_soup.find(
                        string=lambda text: text and 'error(s)' in text
                    )
                    if error_text:
                        warning = error_text.parent.parent.get_text(" ", strip=True)
                        meta_check_df.loc[i-1, 'SCHEV Warning'] = warning

                # Convert to absolute
                base = "https://portals.schev.edu/institutions/PUBLIC/UVA/"
                table_url = urljoin(base, relative_url)
                # print("Full URL:", table_url)

                driver.get(table_url)
                time.sleep(1) # wait for page to load; adjust as needed
                table_page_html = driver.page_source

                # Parse tables with pandas
                tables = pd.read_html(StringIO(table_page_html), header=0)
                df = tables[0]  # Get the first table, adjust index if needed

                # sometimes it's socsec1, sometimes SOCSEC1. Capitalize it if needed.
                df.columns = [str(c).upper() if str(c).lower() == 'socsec1' else c for c in df.columns]

                if error_code == "BFA010W10":  #fix SCHEV table error 
                    df = df.rename(columns={'Note2': 'SOCSEC1'})
                # all_error_tables_for_excel.append(df) Moved below if column below. How was it even working here???

                if 'SOCSEC1' in df.columns:
                    print('in socsec1')
                    df['SOCSEC1'] = df['SOCSEC1'].astype('string').str.strip()
                    VCSIN_to_SSID['VCSIN'] = VCSIN_to_SSID['VCSIN'].astype('string').str.strip()

                    df = df.merge(
                        VCSIN_to_SSID[['VCSIN', 'Student System ID']],
                        how='left',
                        left_on='SOCSEC1',
                        right_on='VCSIN'
                    ).drop(columns='VCSIN')
                    print(df.head(10))
                    print('out of socsec1')


                df[f'Comments V{version_number}'] = ''
                all_error_tables_for_excel.append(df)
            else:
                print("No match found.")
        except Exception as e:
            df = pd.DataFrame({
                'ERROR': [f"Could not retrieve SCHEV table for {error_code}"],
                'DETAILS': [str(e)]
            })

            all_error_tables_for_excel.append(df)
            print(f"Error visiting {link}: {e}")

    print("\nDone visiting all links.")

    print("\nSaving to Excel...")
    
    with pd.ExcelWriter(excel_output_path, engine='openpyxl') as writer:
        for i, df in enumerate(all_error_tables_for_excel):
            sheet_name = all_error_table_names_for_excel[i]
            
            # Write each DataFrame to a separate sheet
            df.to_excel(writer, sheet_name=sheet_name, index=False,startrow=3)
            
            # Put description and link in the first two rows
            try:

                # meta_check_df.loc[i-1, 'SCHEV Definition']
                description = meta_check_df.loc[meta_check_df['Err'] == sheet_name]['SCHEV Definition'].values[0]
                warning = meta_check_df.loc[meta_check_df['Err'] == sheet_name]['SCHEV Warning'].values[0]
                # link = meta_data.loc[meta_data['ErrCode'] == sheet_name]['Link'].values[0]
                link = meta_check_df.loc[meta_check_df['Err'] == sheet_name]['url'].values[0]
                worksheet = writer.sheets[sheet_name]
                worksheet.cell(row=2, column=2).value = f"Definition: {description}"
                worksheet.cell(row=1, column=1).value = f"{warning}"
                worksheet.cell(row=3, column=3).value = f"Link: {link}"
            except Exception as e:
                print(f"Error adding metadata for sheet {sheet_name}: {e}")
    print("\nDone.")
    driver.quit()
else:
    # Find which new error codes are not in meta_data and send message to add them
    for error in meta_check_df['Err']:
        if error not in meta_data['ErrCode'].tolist():
            print(f"Error code {error} not found in meta_data. Please add and re-run. '" + meta_data_path +"'")


Browser opened. Please log in manually.


InvalidSessionIdException: Message: invalid session id: session deleted as the browser has closed the connection
from disconnected: not connected to DevTools
  (Session info: chrome=157.0.8103.39)
Stacktrace:
Symbols not available. Dumping unresolved backtrace:
	(No symbol) [0x7ff712d09cf5]
	(No symbol) [0x7ff712c09540]
	(No symbol) [0x7ff712a92763]
	(No symbol) [0x7ff712a87c5f]
	(No symbol) [0x7ff712a988e9]
	(No symbol) [0x7ff712ac0f56]
	(No symbol) [0x7ff712ac51bf]
	(No symbol) [0x7ff712aab389]
	(No symbol) [0x7ff712aac6b3]
	(No symbol) [0x7ff7131d52e1]
	(No symbol) [0x7ff7131fc54b]
	(No symbol) [0x7ff7131f0c00]
	(No symbol) [0x7ff712e32f3e]
	(No symbol) [0x7ff712c18d85]
	(No symbol) [0x7ff712c13424]
	(No symbol) [0x7ff712c135b4]
	(No symbol) [0x7ff712bfc7ec]
	(No symbol) [0x7ffd376edda7]
	(No symbol) [0x7ffd38a7ee4c]
